In [ ]:
import io
import re
import zstandard as zstd

from pathlib import Path

import chess.pgn
import numpy as np
import pandas as pd

In [ ]:
DATA_DIR = Path("data")

PGN_ZST_PATH = DATA_DIR / "lichess_db_april_2017_eval.pgn.zst"

MAX_GAMES = 50_000

In [ ]:
# Small helpers for eval and clock parsing
EVAL_RE = re.compile(r"\[%eval\s+([^\]]+)\]")
CLK_RE = re.compile(r"\[%clk\s+([^\]]+)\]")

def parse_clock_seconds(clock_text):
  if clock_text is None:
    return np.nan

  parts = clock_text.strip().split(":")

  if len(parts) != 3:
    return np.nan

  hours, minutes, seconds = parts

  try:
    total = 3600 * int(hours)
    total += 60 * int(minutes)
    total += float(seconds)
    return total
  except ValueError:
    return np.nan
  
def parse_eval_cp(eval_text):
  if eval_text is None:
    return np.nan

  eval_text = eval_text.strip()

  if eval_text.startswith("#"):
    mate_text = eval_text.replace("#", "")

    try:
      mate = int(mate_text)
    except ValueError:
      return np.nan

    sign = 1 if mate > 0 else -1
    capped_mate_cp = 10_000 - 100 * abs(mate)

    return sign * capped_mate_cp

  try:
    return 100.0 * float(eval_text)
  except ValueError:
    return np.nan
  
def get_comment_eval(comment):
  match = EVAL_RE.search(comment)

  if match is None:
    return np.nan

  return parse_eval_cp(match.group(1))

def get_comment_clock(comment):
  match = CLK_RE.search(comment)

  if match is None:
    return np.nan

  return parse_clock_seconds(match.group(1))

In [ ]:
def iter_pgn_games_zst(path, max_games=None):
  count = 0

  with open(path, "rb") as compressed:
    dctx = zstd.ZstdDecompressor()
    stream = dctx.stream_reader(compressed)

    text_stream = io.TextIOWrapper(
      stream,
      encoding="utf-8",
      errors="replace",
    )

    while True:
      game = chess.pgn.read_game(text_stream)

      if game is None:
        break

      yield game
      count += 1

      if max_games is not None and count >= max_games:
        break

In [ ]:
def safe_int(value):
  try:
    return int(value)
  except (TypeError, ValueError):
    return np.nan

In [ ]:
def result_to_scores(result):
  if result == "1-0":
    return 1.0, 0.0

  if result == "0-1":
    return 0.0, 1.0

  if result == "1/2-1/2":
    return 0.5, 0.5

  return np.nan, np.nan

In [ ]:
def extract_game_metadata(game, game_id):
  headers = game.headers

  white_score, black_score = result_to_scores(headers.get("Result"))

  return {
    "game_id": game_id,
    "site": headers.get("Site"),
    "utc_date": headers.get("UTCDate"),
    "utc_time": headers.get("UTCTime"),
    "event": headers.get("Event"),
    "white": headers.get("White"),
    "black": headers.get("Black"),
    "white_elo": safe_int(headers.get("WhiteElo")),
    "black_elo": safe_int(headers.get("BlackElo")),
    "white_rating_diff": safe_int(headers.get("WhiteRatingDiff")),
    "black_rating_diff": safe_int(headers.get("BlackRatingDiff")),
    "result": headers.get("Result"),
    "white_score": white_score,
    "black_score": black_score,
    "eco": headers.get("ECO"),
    "opening": headers.get("Opening"),
    "time_control": headers.get("TimeControl"),
    "termination": headers.get("Termination"),
  }

In [ ]:
def move_phase(ply):
  if ply <= 20:
    return "opening"

  if ply <= 60:
    return "middlegame"

  return "endgame"

In [ ]:
def extract_move_rows(game, game_id):
  rows = []

  board = game.board()
  prev_eval_cp = np.nan
  node = game

  for ply, next_node in enumerate(game.mainline(), start=1):
    move = next_node.move
    mover_color = board.turn
    mover = "white" if mover_color == chess.WHITE else "black"

    san = board.san(move)
    board.push(move)

    comment = next_node.comment or ""
    eval_cp = get_comment_eval(comment)
    clock_seconds = get_comment_clock(comment)

    if np.isnan(prev_eval_cp) or np.isnan(eval_cp):
      eval_delta_cp = np.nan
      mover_loss_cp = np.nan
    else:
      eval_delta_cp = eval_cp - prev_eval_cp

      if mover == "white":
        mover_loss_cp = -eval_delta_cp
      else:
        mover_loss_cp = eval_delta_cp

      mover_loss_cp = max(0.0, mover_loss_cp)

    rows.append({
      "game_id": game_id,
      "ply": ply,
      "move_number": (ply + 1) // 2,
      "mover": mover,
      "san": san,
      "uci": move.uci(),
      "phase": move_phase(ply),
      "eval_cp": eval_cp,
      "prev_eval_cp": prev_eval_cp,
      "eval_delta_cp": eval_delta_cp,
      "mover_loss_cp": mover_loss_cp,
      "clock_seconds": clock_seconds,
    })

    prev_eval_cp = eval_cp
    node = next_node

  return rows

In [ ]:
def parse_pgn_zst(path, max_games=None):
  game_rows = []
  move_rows = []

  for game_id, game in enumerate(iter_pgn_games_zst(path, max_games)):
    game_rows.append(extract_game_metadata(game, game_id))
    move_rows.extend(extract_move_rows(game, game_id))

    if game_id > 0 and game_id % 1000 == 0:
      print(f"Parsed {game_id:,} games")

  games_df = pd.DataFrame(game_rows)
  moves_df = pd.DataFrame(move_rows)

  return games_df, moves_df

In [ ]:
games_df, moves_df = parse_pgn_zst(
  PGN_ZST_PATH,
  max_games=MAX_GAMES,
)

In [ ]:
games_df.head()

In [ ]:
moves_df.head()

In [ ]:
print("Number of games:", len(games_df))
print("Number of moves:", len(moves_df))

print()
print("Games with White Elo:", games_df["white_elo"].notna().mean())
print("Games with Black Elo:", games_df["black_elo"].notna().mean())

print()
print("Moves with eval:", moves_df["eval_cp"].notna().mean())
print("Moves with clock:", moves_df["clock_seconds"].notna().mean())

In [ ]:
games_df[[
  "white_elo",
  "black_elo",
  "time_control",
  "termination",
]].describe(include="all")

In [ ]:
moves_df[[
  "ply",
  "eval_cp",
  "mover_loss_cp",
  "clock_seconds",
]].describe()

In [ ]:
def add_player_columns(moves_df, games_df):
  game_cols = [
    "game_id",
    "white",
    "black",
    "white_elo",
    "black_elo",
    "white_score",
    "black_score",
    "time_control",
    "termination",
  ]

  df = moves_df.merge(
    games_df[game_cols],
    on="game_id",
    how="left",
  )

  is_white = df["mover"] == "white"

  df["player"] = np.where(is_white, df["white"], df["black"])
  df["opponent"] = np.where(is_white, df["black"], df["white"])

  df["player_elo"] = np.where(
    is_white,
    df["white_elo"],
    df["black_elo"],
  )

  df["opponent_elo"] = np.where(
    is_white,
    df["black_elo"],
    df["white_elo"],
  )

  df["player_score"] = np.where(
    is_white,
    df["white_score"],
    df["black_score"],
  )

  return df

In [ ]:
moves_player_df = add_player_columns(moves_df, games_df)
moves_player_df.head()

In [ ]:
def add_move_quality_flags(df):
  df = df.copy()

  loss = df["mover_loss_cp"]

  df["is_inaccuracy"] = loss >= 50
  df["is_mistake"] = loss >= 100
  df["is_blunder"] = loss >= 300
  df["is_major_blunder"] = loss >= 500

  df["is_low_time"] = df["clock_seconds"] <= 10
  df["is_very_low_time"] = df["clock_seconds"] <= 5

  return df

In [ ]:
moves_player_df = add_move_quality_flags(moves_player_df)
moves_player_df.head()

In [ ]:
def q10(x):
  return x.quantile(0.10)

In [ ]:
def q90(x):
  return x.quantile(0.90)

In [ ]:
def aggregate_player_game_features(df):
  group_cols = [
    "game_id",
    "player",
    "opponent",
    "player_elo",
    "opponent_elo",
    "player_score",
    "time_control",
  ]

  agg = df.groupby(group_cols).agg(
    n_moves=("ply", "size"),
    mean_loss_cp=("mover_loss_cp", "mean"),
    median_loss_cp=("mover_loss_cp", "median"),
    q90_loss_cp=("mover_loss_cp", q90),
    max_loss_cp=("mover_loss_cp", "max"),
    std_loss_cp=("mover_loss_cp", "std"),
    inaccuracy_rate=("is_inaccuracy", "mean"),
    mistake_rate=("is_mistake", "mean"),
    blunder_rate=("is_blunder", "mean"),
    major_blunder_rate=("is_major_blunder", "mean"),
    mean_clock_seconds=("clock_seconds", "mean"),
    min_clock_seconds=("clock_seconds", "min"),
    low_time_rate=("is_low_time", "mean"),
    very_low_time_rate=("is_very_low_time", "mean"),
  )

  agg = agg.reset_index()

  return agg

In [ ]:
player_game_df = aggregate_player_game_features(moves_player_df)
player_game_df.head()

In [ ]:
def aggregate_phase_features(df):
  group_cols = [
    "game_id",
    "player",
  ]

  phase_agg = df.pivot_table(
    index=group_cols,
    columns="phase",
    values="mover_loss_cp",
    aggfunc=["mean", "median"],
  )

  phase_agg.columns = [
    f"{stat}_{phase}_loss_cp"
    for stat, phase in phase_agg.columns
  ]

  phase_agg = phase_agg.reset_index()

  return phase_agg

In [ ]:
phase_df = aggregate_phase_features(moves_player_df)
phase_df.head()

In [ ]:
player_game_df = player_game_df.merge(
  phase_df,
  on=["game_id", "player"],
  how="left",
)

In [ ]:
def add_player_game_order(df):
  df = df.copy()

  df = df.sort_values([
    "player",
    "game_id",
  ])

  df["player_game_index"] = df.groupby("player").cumcount()

  return df

In [ ]:
player_game_df = add_player_game_order(player_game_df)
player_game_df.head()

In [ ]:
player_game_df = add_player_game_order(player_game_df)
player_game_df.head()

In [ ]:
def assign_game_windows(df, games_per_window):
  df = df.copy()

  df["window_id"] = (
    df["player_game_index"] // games_per_window
  )

  return df

In [ ]:
GAMES_PER_WINDOW = 10

player_game_windowed_df = assign_game_windows(
  player_game_df,
  games_per_window=GAMES_PER_WINDOW,
)

In [ ]:
def aggregate_player_windows(df):
  group_cols = [
    "player",
    "window_id",
    "time_control",
  ]

  feature_cols = [
    "n_moves",
    "mean_loss_cp",
    "median_loss_cp",
    "q90_loss_cp",
    "max_loss_cp",
    "std_loss_cp",
    "inaccuracy_rate",
    "mistake_rate",
    "blunder_rate",
    "major_blunder_rate",
    "mean_clock_seconds",
    "min_clock_seconds",
    "low_time_rate",
    "very_low_time_rate",
    "mean_opening_loss_cp",
    "mean_middlegame_loss_cp",
    "mean_endgame_loss_cp",
    "median_opening_loss_cp",
    "median_middlegame_loss_cp",
    "median_endgame_loss_cp",
  ]

  available_features = [
    col for col in feature_cols
    if col in df.columns
  ]

  agg_dict = {
    "game_id": "nunique",
    "player_elo": "mean",
    "opponent_elo": "mean",
    "player_score": "mean",
  }

  for col in available_features:
    agg_dict[col] = "mean"

  out = df.groupby(group_cols).agg(agg_dict)
  out = out.reset_index()

  out = out.rename(columns={
    "game_id": "n_games",
    "player_elo": "target_elo",
    "opponent_elo": "mean_opponent_elo",
    "player_score": "mean_score",
  })

  return out

In [ ]:
player_window_df = aggregate_player_windows(
  player_game_windowed_df,
)

In [ ]:
player_window_df.head()

In [ ]:
model_df = player_window_df[
  player_window_df["n_games"] == GAMES_PER_WINDOW
].copy()

In [ ]:
print("Player-window rows:", len(model_df))
print("Unique players:", model_df["player"].nunique())

In [ ]:
model_df[[
  "target_elo",
  "mean_opponent_elo",
  "mean_loss_cp",
  "blunder_rate",
  "mean_clock_seconds",
]].describe()

In [ ]:
OUT_DIR = Path("processed")
OUT_DIR.mkdir(exist_ok=True)

In [ ]:
games_df.to_parquet(OUT_DIR / "games.parquet", index=False)
moves_player_df.to_parquet(
  OUT_DIR / "moves_player.parquet",
  index=False,
)
player_game_df.to_parquet(
  OUT_DIR / "player_game.parquet",
  index=False,
)
model_df.to_parquet(
  OUT_DIR / "player_windows.parquet",
  index=False,
)

In [ ]:
import matplotlib.pyplot as plt

In [ ]:
plt.figure(figsize=(7, 4))
plt.hist(model_df["target_elo"], bins=50)
plt.xlabel("Target Elo")
plt.ylabel("Player-window count")
plt.title("Target Elo distribution")
plt.show()

In [ ]:
plt.figure(figsize=(7, 4))
plt.scatter(
  model_df["target_elo"],
  model_df["mean_loss_cp"],
  alpha=0.2,
)
plt.xlabel("Target Elo")
plt.ylabel("Mean centipawn loss")
plt.title("Mean loss versus Elo")
plt.show()

In [ ]:
plt.figure(figsize=(7, 4))
plt.scatter(
  model_df["target_elo"],
  model_df["blunder_rate"],
  alpha=0.2,
)
plt.xlabel("Target Elo")
plt.ylabel("Blunder rate")
plt.title("Blunder rate versus Elo")
plt.show()

In [ ]:
style_counts = (
  model_df
  .groupby("time_control")
  .size()
  .reset_index(name="n_rows")
  .sort_values("n_rows", ascending=False)
)

style_counts.head(20)

In [ ]:
most_common_time_control = (
  model_df["time_control"]
  .value_counts()
  .idxmax()
)

most_common_time_control

In [ ]:
style_df = model_df[
  model_df["time_control"] == most_common_time_control
].copy()

print("Selected time control:", most_common_time_control)
print("Rows:", len(style_df))
print("Unique players:", style_df["player"].nunique())

In [ ]:
style_df[[
  "target_elo",
  "mean_opponent_elo",
  "mean_loss_cp",
  "median_loss_cp",
  "q90_loss_cp",
  "blunder_rate",
  "mistake_rate",
  "mean_clock_seconds",
]].describe()

In [ ]:
plt.figure(figsize=(7, 4))
plt.hist(style_df["target_elo"], bins=40)
plt.xlabel("Target Elo")
plt.ylabel("Player-window count")
plt.title(f"Elo distribution for {most_common_time_control}")
plt.show()

In [ ]:
plt.figure(figsize=(7, 4))
plt.scatter(
  style_df["target_elo"],
  style_df["mean_loss_cp"],
  alpha=0.35,
)
plt.xlabel("Target Elo")
plt.ylabel("Mean centipawn loss")
plt.title("Mean centipawn loss versus Elo")
plt.show()

In [ ]:
plt.figure(figsize=(7, 4))
plt.scatter(
  style_df["target_elo"],
  style_df["blunder_rate"],
  alpha=0.35,
)
plt.xlabel("Target Elo")
plt.ylabel("Blunder rate")
plt.title("Blunder rate versus Elo")
plt.show()

In [ ]:
candidate_features = [
  "n_games",
  "n_moves",
  "mean_opponent_elo",
  "mean_score",
  "mean_loss_cp",
  "median_loss_cp",
  "q90_loss_cp",
  "max_loss_cp",
  "std_loss_cp",
  "inaccuracy_rate",
  "mistake_rate",
  "blunder_rate",
  "major_blunder_rate",
  "mean_clock_seconds",
  "min_clock_seconds",
  "low_time_rate",
  "very_low_time_rate",
  "mean_opening_loss_cp",
  "mean_middlegame_loss_cp",
  "mean_endgame_loss_cp",
]

In [ ]:
feature_cols = [
  col for col in candidate_features
  if col in style_df.columns
]

feature_cols

In [ ]:
model_ready_df = style_df[
  ["player", "target_elo"] + feature_cols
].copy()

model_ready_df = model_ready_df.replace([np.inf, -np.inf], np.nan)
model_ready_df = model_ready_df.dropna()

print("Rows after dropping NaNs:", len(model_ready_df))
print("Unique players:", model_ready_df["player"].nunique())

In [ ]:
from sklearn.model_selection import train_test_split

In [ ]:
unique_players = (
  model_ready_df["player"]
  .dropna()
  .astype(str)
  .unique()
)

unique_players = np.array(unique_players)

train_players, test_players = train_test_split(
  unique_players,
  test_size=0.25,
  random_state=42,
)

train_df = model_ready_df[
  model_ready_df["player"].isin(train_players)
].copy()

test_df = model_ready_df[
  model_ready_df["player"].isin(test_players)
].copy()

print("Train rows:", len(train_df))
print("Test rows:", len(test_df))
print("Train players:", train_df["player"].nunique())
print("Test players:", test_df["player"].nunique())

In [ ]:
X_train = train_df[feature_cols]
y_train = train_df["target_elo"]

X_test = test_df[feature_cols]
y_test = test_df["target_elo"]

In [ ]:
from sklearn.metrics import mean_absolute_error
from sklearn.metrics import mean_squared_error

In [ ]:
baseline_pred = np.full(
  shape=len(y_test),
  fill_value=y_train.mean(),
)

baseline_mae = mean_absolute_error(y_test, baseline_pred)

baseline_mse = mean_squared_error(
  y_test,
  baseline_pred,
)

baseline_rmse = np.sqrt(baseline_mse)

print("Baseline MAE:", baseline_mae)
print("Baseline RMSE:", baseline_rmse)

In [ ]:
from sklearn.linear_model import Ridge
from sklearn.pipeline import make_pipeline
from sklearn.preprocessing import StandardScaler

In [ ]:
ridge_model = make_pipeline(
  StandardScaler(),
  Ridge(alpha=10.0),
)

ridge_model.fit(X_train, y_train)

ridge_pred = ridge_model.predict(X_test)

ridge_mae = mean_absolute_error(y_test, ridge_pred)
ridge_mse = mean_squared_error(
  y_test,
  ridge_pred,
)
ridge_rmse = np.sqrt(ridge_mse)

print("Ridge MAE:", ridge_mae)
print("Ridge RMSE:", ridge_rmse)

In [ ]:
from sklearn.ensemble import RandomForestRegressor

In [ ]:
rf_model = RandomForestRegressor(
  n_estimators=300,
  max_depth=5,
  min_samples_leaf=10,
  random_state=42,
  n_jobs=-1,
)

rf_model.fit(X_train, y_train)

rf_pred = rf_model.predict(X_test)

rf_mae = mean_absolute_error(y_test, rf_pred)
rf_mse = mean_squared_error(
  y_test,
  rf_pred,
)
rf_rmse = np.sqrt(rf_mse)

print("Random forest MAE:", rf_mae)
print("Random forest RMSE:", rf_rmse)

In [ ]:
results_df = pd.DataFrame({
  "model": [
    "mean_baseline",
    "ridge",
    "random_forest",
  ],
  "mae": [
    baseline_mae,
    ridge_mae,
    rf_mae,
  ],
  "rmse": [
    baseline_rmse,
    ridge_rmse,
    rf_rmse,
  ],
})

results_df

In [ ]:
plt.figure(figsize=(7, 4))
plt.bar(results_df["model"], results_df["mae"])
plt.ylabel("MAE [Elo]")
plt.title("Toy model comparison")
plt.xticks(rotation=30, ha="right")
plt.show()

In [ ]:
plt.figure(figsize=(5, 5))
plt.scatter(y_test, ridge_pred, alpha=0.5)
plt.xlabel("True Elo")
plt.ylabel("Predicted Elo")
plt.title("Ridge: predicted versus true Elo")

lims = [
  min(y_test.min(), ridge_pred.min()),
  max(y_test.max(), ridge_pred.max()),
]

plt.plot(lims, lims)
plt.show()

In [ ]:
plt.figure(figsize=(5, 5))
plt.scatter(y_test, rf_pred, alpha=0.5)
plt.xlabel("True Elo")
plt.ylabel("Predicted Elo")
plt.title("Random forest: predicted versus true Elo")

lims = [
  min(y_test.min(), rf_pred.min()),
  max(y_test.max(), rf_pred.max()),
]

plt.plot(lims, lims)
plt.show()

In [ ]:
test_eval_df = test_df.copy()
test_eval_df["ridge_pred"] = ridge_pred
test_eval_df["rf_pred"] = rf_pred

test_eval_df["ridge_residual"] = (
  test_eval_df["target_elo"] - test_eval_df["ridge_pred"]
)

test_eval_df["rf_residual"] = (
  test_eval_df["target_elo"] - test_eval_df["rf_pred"]
)

In [ ]:
plt.figure(figsize=(7, 4))
plt.scatter(
  test_eval_df["target_elo"],
  test_eval_df["rf_residual"],
  alpha=0.5,
)
plt.axhline(0)
plt.xlabel("True Elo")
plt.ylabel("Residual: true - predicted")
plt.title("Random forest residuals")
plt.show()

In [ ]:
importance_df = pd.DataFrame({
  "feature": feature_cols,
  "importance": rf_model.feature_importances_,
})

importance_df = importance_df.sort_values(
  "importance",
  ascending=False,
)

importance_df

In [ ]:
plt.figure(figsize=(7, 5))
plt.barh(
  importance_df["feature"].head(15)[::-1],
  importance_df["importance"].head(15)[::-1],
)
plt.xlabel("Random forest feature importance")
plt.title("Top feature importances")
plt.show()

In [ ]:
feature_cols_no_opp = [
  col for col in feature_cols
  if col != "mean_opponent_elo"
]

In [ ]:
X_train_no_opp = train_df[feature_cols_no_opp]
X_test_no_opp = test_df[feature_cols_no_opp]

In [ ]:
rf_no_opp_model = RandomForestRegressor(
  n_estimators=300,
  max_depth=5,
  min_samples_leaf=10,
  random_state=42,
  n_jobs=-1,
)

rf_no_opp_model.fit(X_train_no_opp, y_train)

rf_no_opp_pred = rf_no_opp_model.predict(X_test_no_opp)

rf_no_opp_mae = mean_absolute_error(y_test, rf_no_opp_pred)
rf_no_opp_mse = mean_squared_error(
  y_test,
  rf_no_opp_pred,
)
rf_no_opp_rmse = np.sqrt(rf_no_opp_mse)

print("RF without opponent Elo MAE:", rf_no_opp_mae)
print("RF without opponent Elo RMSE:", rf_no_opp_rmse)

In [ ]:
pd.DataFrame({
  "model": [
    "mean_baseline",
    "rf_with_opponent_elo",
    "rf_without_opponent_elo",
  ],
  "mae": [
    baseline_mae,
    rf_mae,
    rf_no_opp_mae,
  ],
  "rmse": [
    baseline_rmse,
    rf_rmse,
    rf_no_opp_rmse,
  ],
})